<a href="https://colab.research.google.com/github/zaetae/regime-aware-ml-trading/blob/main/33_intraday_hourly_pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 33. Intraday pilot: the pipeline on hourly bars

**Question from the supervisor's comments:** does the event-driven approach also work on intraday data, where there would be more events per instrument?

**Design**
- Same 24 tickers, same detectors, features, triple-barrier labels (2 ATR take-profit, 2 ATR stop-loss, 10 bars) and pooled Random Forest as in the daily notebooks. Only the bars change: one bar is now one hour (regular trading hours, 7 bars per day), so windows such as the 50-bar extreme or the 10-bar holding limit are measured in hours.
- Data: Yahoo Finance serves hourly bars for about the last 730 days only. This is therefore a **pilot on about two years**, not a long history, and the number of events is similar to the daily data, not larger.
- Split: events in time order, first 60 percent for training, last 30 percent for testing, 10 percent purged in between. No settings are tuned. Random baseline: stratified guessing (20 draws). Benchmark: trade every event in its intended direction. Cost 1.4 bps round trip.
- Positions can stay open over the night (the 10-bar limit is about a day and a half), and entry is at the close of the event bar, as in the daily tests.

In [ ]:
import sys, os
for m in list(sys.modules.keys()):
    if m.startswith('src'):
        del sys.modules[m]

REPO_DIR  = '/content/regime-aware-ml-trading'
PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
if not os.path.isdir(PROJ_ROOT):
    os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
else:
    os.system(f'cd {REPO_DIR} && git pull -q')
os.system(f'{sys.executable} -m pip install -q yfinance hmmlearn scikit-learn seaborn statsmodels')

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

import pandas as pd, numpy as np, matplotlib.pyplot as plt
print("src exists:", os.path.isdir(os.path.join(PROJ_ROOT, 'src')))

In [ ]:
import yfinance as yf
from src.experiments.helpers import build_event_data
from src.backtest.simulator import simulate_trades
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

ALL = ['SPY','QQQ','NVDA','GOOGL','AAPL','MSFT','AMZN','META','AVGO','TSLA','COST','ADBE',
       'AMD','NFLX','INTC','CSCO','JNJ','KO','WMT','PG','CAT','F','XOM','JPM']
END = pd.Timestamp.today().normalize() + pd.Timedelta(days=1)
START = END - pd.Timedelta(days=725)          # Yahoo serves hourly data for about the last 730 days only

def download_hourly(tk):
    raw = yf.download(tk, start=START.strftime('%Y-%m-%d'), end=END.strftime('%Y-%m-%d'),
                      interval='60m', auto_adjust=False, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(1)
    raw = raw[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
    if raw.index.tz is not None:
        raw.index = raw.index.tz_convert('America/New_York').tz_localize(None)
    raw = raw[~raw.index.duplicated()]
    raw.index.name = 'Date'
    return raw

event_data = {}
for tk in ALL:
    try:
        raw = download_hourly(tk)
        event_data[tk] = build_event_data(tk, raw=raw)
    except Exception as e:
        print(tk, "FAILED:", e)

all_cols = sorted(set().union(*[set(d['features'].columns) for d in event_data.values()]))
parts = []
for tk, d in event_data.items():
    lab = d['labeled_df'].reset_index(drop=True)
    assert len(d['features']) == len(lab) == len(d['labels']), tk
    dates = pd.to_datetime(lab['event_date']).values
    X = d['features'].reindex(columns=all_cols, fill_value=0).fillna(0).reset_index(drop=True)
    meta = pd.DataFrame({'ticker': tk, 'i': np.arange(len(lab)), 'date': dates, 'y': d['labels'].values,
                         'intended': lab['intended_direction'].values})
    parts.append(pd.concat([meta, X], axis=1))
EV = pd.concat(parts, ignore_index=True)
FEATS = all_cols
LAB = {tk: d['labeled_df'].reset_index(drop=True) for tk, d in event_data.items()}
RAW = {tk: d['raw'] for tk, d in event_data.items()}

nb = pd.Series({tk: len(r) for tk, r in RAW.items()})
print("period:", min(r.index.min() for r in RAW.values()).date(), "to", max(r.index.max() for r in RAW.values()).date())
print("tickers:", len(RAW), "| hourly bars per ticker (median):", int(nb.median()), "| events in total:", len(EV),
      "| events per ticker (median):", int(EV.groupby('ticker').size().median()))
print("label mix:", EV.y.value_counts(normalize=True).round(3).to_dict())

## Classification and trading result

In [ ]:
SEEDS = [42, 7, 123, 11, 99]
N_DUMMY = 20
COST = 1.4e-4
def f1m(y, p): return f1_score(y, p, average='macro')

# chronological split on the event dates: first 60 percent train, last 30 percent test, 10 percent purged in between
d_sorted = np.sort(EV.date.values)
T_TRAIN = pd.Timestamp(d_sorted[int(0.60 * len(d_sorted))])
T_TEST = pd.Timestamp(d_sorted[int(0.70 * len(d_sorted))])
train = (EV.date < T_TRAIN).values
test = (EV.date >= T_TEST).values
print(f"train: before {T_TRAIN.date()} ({train.sum()} events) | purge | test: from {T_TEST.date()} ({test.sum()} events)")

Xtr, ytr, Xte, yte = EV.loc[train, FEATS], EV.y[train], EV.loc[test, FEATS], EV.y[test]
rnd = np.mean([f1m(yte, DummyClassifier(strategy='stratified', random_state=k).fit(Xtr, ytr).predict(Xte)) for k in range(N_DUMMY)])
probs = []
for s in SEEDS:
    m = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=s, n_jobs=-1).fit(Xtr, ytr)
    probs.append(pd.DataFrame(m.predict_proba(Xte), columns=m.classes_))
f1s = [f1m(yte, p.idxmax(axis=1).values) for p in probs]
print(f"\nPooled Random Forest on hourly events: F1 macro {np.mean(f1s):.3f} (seeds {min(f1s):.3f} to {max(f1s):.3f}) | random baseline {rnd:.3f} | margin {np.mean(f1s) - rnd:+.3f}")

TE = EV[test].reset_index(drop=True)
def trades_for(pred, pt, sl, rows=TE):
    out = []
    for tk, g in rows.groupby('ticker'):
        t = simulate_trades(RAW[tk], LAB[tk].iloc[g['i'].values], pred[g.index.values], pt_mult=pt, sl_mult=sl,
                            max_holding=10, cost_per_trade=COST)
        if len(t): t['ticker'] = tk; out.append(t)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=['net_return', 'ticker'])

def stats(T):
    r = T.net_return.values
    if len(r) < 2: return dict(trades=len(r), mean_net_pct=np.nan, win_rate=np.nan, t_stat=np.nan, tickers_positive='-')
    pos = T.groupby('ticker').net_return.mean()
    return dict(trades=len(r), mean_net_pct=100 * r.mean(), win_rate=100 * (r > 0).mean(),
                t_stat=r.mean() / (r.std(ddof=1) / np.sqrt(len(r))), tickers_positive=f"{(pos > 0).sum()}/{len(pos)}")

rows, all_trades = [], {}
for pt, sl in [(2, 2), (2, 3)]:
    # model: a prediction from each seed, averaged over seeds by running the trades for every seed
    res = [stats(trades_for(p.idxmax(axis=1).values, pt, sl)) for p in probs]
    avg = {k: (np.nanmean([r[k] for r in res]) if k != 'tickers_positive' else res[0][k]) for k in res[0]}
    rows.append(dict(setting=f'TP {pt} / SL {sl}', strategy='Pooled RF (mean of 5 seeds)', **avg))
    tr_all = trades_for(TE['intended'].values, pt, sl)        # no model: trade every event in its intended direction
    all_trades[(pt, sl)] = tr_all
    rows.append(dict(setting=f'TP {pt} / SL {sl}', strategy='No model, trade every event', **stats(tr_all)))
pd.options.display.float_format = '{:.3f}'.format
TAB = pd.DataFrame(rows)
print("\nHourly bars, test period, cost 1.4 bps round trip")
print(TAB.to_string(index=False))

## Cost compared with the move, per ticker, cumulative return

In [ ]:
# How large is the cost compared with the move being traded? Take-profit distance in basis points (2 ATR / close).
tp_bps = []
for tk, r in RAW.items():
    h, l, c = r.High, r.Low, r.Close
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    tp_bps.append((2 * tr.rolling(14).mean() / c * 1e4).median())
print(f"Median take-profit distance (2 ATR, hourly): {np.median(tp_bps):.0f} bps; round-trip cost 1.4 bps = "
      f"{1.4 / np.median(tp_bps) * 100:.1f}% of it")

# per-ticker view of the model trades (seed 42) and the no-model benchmark
p0 = probs[0].idxmax(axis=1).values
Tm = trades_for(p0, 2, 2); Tn = all_trades[(2, 2)]
cmp = pd.DataFrame({'model': Tm.groupby('ticker').net_return.mean() * 100,
                    'no model': Tn.groupby('ticker').net_return.mean() * 100,
                    'model trades': Tm.groupby('ticker').size()}).sort_values('model', ascending=False)
print("\nMean net % per trade by ticker (TP 2 / SL 2, seed 42)")
print(cmp.round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
cum = Tm.sort_values('event_date').net_return.cumsum() * 100
ax[0].plot(pd.to_datetime(Tm.sort_values('event_date').event_date), cum.values, color='#4C72B0', label='model')
cn = Tn.sort_values('event_date').net_return.cumsum() * 100
ax[0].plot(pd.to_datetime(Tn.sort_values('event_date').event_date), cn.values, color='#8C8C8C', label='no model')
ax[0].set_title('Cumulative sum of net returns per trade (%), hourly test period'); ax[0].legend(); ax[0].tick_params(axis='x', rotation=30)
ax[1].bar(['Hourly: model', 'Hourly: no model'], [Tm.net_return.mean() * 100, Tn.net_return.mean() * 100], color=['#4C72B0', '#8C8C8C'])
ax[1].axhline(0, color='k', lw=0.6); ax[1].set_title('Mean net return per trade (%)')
plt.tight_layout(); plt.show()